# 03 - Data Preparation
**CRISP-DM phase:** Data Preparation

This notebook turns the raw NHTSA files into the two analytical tables used for modelling:

| Output | One row per | Used by |
|---|---|---|
| `data/processed/crss_crash_features.parquet` | sampled crash (CRSS 2020-2024) | S1 injury, S2 serious/fatal |
| `data/processed/complaints_model.parquet` | owner complaint (`ODINO`) | U1 component routing, U2 serious-incident flag |

The full build is implemented in `src/build_crss_features.py` and `src/build_complaints.py`. Each step below shows the decision, the code behind it, and a check on the result. Raw files in `data/raw` are never modified.

## 3.0 Setup
Load the project paths and the two preparation modules. `RUN_FULL_BUILD = False` reuses the prepared tables (the full rebuild from raw data takes about 20 minutes); set it to `True` to reproduce everything from the raw files.

In [1]:
from pathlib import Path
import csv, io, json, sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from project_paths import (OUTPUT_DIR, CRSS_TEST_YEAR, CRSS_TRAIN_YEARS, CRSS_VALIDATION_YEAR, PROCESSED_DIR, TABLE_DIR,
                           complaint_file, crss_extracted_dir)
import build_crss_features as crss_build
import build_complaints as complaint_build

RUN_FULL_BUILD = False
pd.set_option("display.max_columns", 50, "display.width", 200)
print("Python:", sys.executable)

Python: C:\Users\delan\Downloads\NHTSA_Road_Vehicle_Safety_Capstone_Last_Lap\.venv\Scripts\python.exe


## 3.1 Build (or reuse) the prepared tables

In [2]:
if RUN_FULL_BUILD:
    crss_build.main()        # ~4 minutes: all 28 CRSS tables x 5 years -> one row per crash
    complaint_build.main()   # ~15 minutes: 2.25M complaint rows -> one row per complaint
for f in ["crss_crash_features.parquet", "complaints_model.parquet"]:
    p = PROCESSED_DIR / f
    print(f"{f:32s} {p.stat().st_size / 1e6:7.1f} MB   built {pd.Timestamp(p.stat().st_mtime, unit='s'):%Y-%m-%d %H:%M}")

crss_crash_features.parquet         47.5 MB   built 2026-10-03 10:28
complaints_model.parquet           446.4 MB   built 2026-10-01 04:38


# Part A - CRSS crash data (structured)

## 3.2 Tables and their grain
CRSS publishes 28 related tables per year; 26 describe the crash, its vehicles and its people, and two (`vpicdecode`, `vpictrailerdecode`) hold manufacturer specifications decoded from each vehicle's VIN. They describe different things: the crash itself, each vehicle, each person, and many multi-row risk-factor lists. Before combining them we need each table's **grain** (what one row represents), because joining tables of different grain directly would duplicate crashes.

In [3]:
inv = pd.read_csv(TABLE_DIR / "crss_raw_file_inventory.csv")
grain = pd.Series(crss_build.TABLE_LEVEL, name="grain")
rows_2024 = inv[inv.year == 2024].set_index("table")["rows"]
overview = pd.DataFrame({"grain": grain, "rows_2024": rows_2024.reindex(grain.index)})
overview.sort_values(["grain", "rows_2024"], ascending=[True, False])

,grain,rows_2024
cevent,crash,84081
weather,crash,52032
crashrf,crash,51817
accident,crash,51658
personrf,person,126167
person,person,126159
nmcrash,person,7469
nmprior,person,5355
nmimpair,person,5256
nmdistract,person,5250


**Interpretation.** 4 tables are at crash level, 16 at vehicle level and 8 at person level (`parkwork`/`pvehiclesf` describe parked vehicles; `vpicdecode`/`vpictrailerdecode` give each vehicle's and trailer's manufacturer specification). The modelling unit is the crash, so vehicle- and person-level information must be **summarised per crash** before it is joined (step 3.6).

> **Initial plan:** build crash features from the 26 tables that describe the crash, its vehicles and its people.
> **What we found:** the two VIN-decode tables were left out, although they hold about 200 specification fields per vehicle, including fitted safety systems (ESC, ABS, forward-collision warning, air-bag locations), weight and engine power. Excluding them would be a manual selection of fields.
> **Decision:** all 28 tables are used. Coded specification fields (`...ID`) are treated like other coded fields, measurements as numeric fields, and the text labels that duplicate the codes are excluded and logged.

## 3.3 Fields excluded from the predictors
At this stage every documented field is considered and none is hand-picked; the fields the final models use are selected later, on evidence (notebook 04, Part B). A field is removed here only for one of four reasons, and every removal is recorded:

* **Leakage:** the field encodes the outcome we are predicting (e.g. maximum injury severity, injury counts, hospital transport).
* **Sampling design:** survey-design variables. `STRATUM` is included here because CRSS assigns strata partly by injury severity, so it would leak the answer.
* **Identifiers:** VINs, carrier IDs and ZIP codes identify records but carry no general pattern.
* **Keys:** join keys such as `CASENUM` and `VEH_NO`.
* **Text labels (VIN-decode tables):** text columns that repeat a coded `...ID` column (e.g. `ForwardCollisionWarning` beside `ForwardCollisionWarningId`) or are free-text notes. The coded version is used.

In [4]:
excluded = pd.read_csv(TABLE_DIR / "crss_excluded_fields.csv").drop_duplicates("field")
def category(row):
    if row.field in crss_build.LEAKAGE: return "1 Leakage (encodes the outcome)"
    if row.field in crss_build.DESIGN: return "2 Sampling design / split variable"
    if row.field in crss_build.KEYS: return "4 Join key"
    if row.reason.startswith("text label"): return "5 Text label of a coded field (code used instead)"
    return "3 Identifier / duplicate / no general meaning"
excluded["category"] = excluded.apply(category, axis=1)
display(excluded.groupby("category").field.agg(["count", lambda s: ", ".join(sorted(s))]).rename(columns={"<lambda_0>": "fields"}))

,count,fields
category,,
1 Leakage (encodes the outcome),12,"HOSPITAL, INJSEV_IM, INJ_SEV, MAXSEV_IM, MAX_S..."
2 Sampling design / split variable,7,"PJ, PSU, PSUSTRAT, PSU_VAR, STRATUM, WEIGHT, YEAR"
3 Identifier / duplicate / no general meaning,38,"DISPLACEMENTCC, DISPLACEMENTCI, DR_ZIP, HAZ_CN..."
4 Join key,8,"CASENUM, EVENTNUM, PER_NO, TRAILER_NO, VEH_NO,..."
5 Text label of a coded field (code used instead),90,"ACTIVESAFETYSYSNOTE, ADAPTIVECRUISECONTROL, AD..."


**Interpretation.** 155 distinct fields are excluded: 12 for leakage, 7 sampling-design fields, 8 join keys, 38 identifiers or exact duplicates, and 90 text labels whose coded version is kept. Every other documented field from all 28 tables remains a candidate for the benchmark models.

### Field accounting: from 1,114 raw fields to 303
Counted per table (CASENUM, for example, counts 28 times because it appears in all 28 tables), every field in the raw tables has one role. The 394 excluded entries above are 155 distinct field names; 417 more repeat information already kept; 303 are used to build the crash features.

In [5]:
funnel = pd.read_csv(OUTPUT_DIR / "selection" / "crss_field_funnel.csv").fillna("")
funnel.iloc[:4]

,step,fields,note
0,Raw fields in the 28 CRSS tables (counted per ...,1114,
1,"Excluded by rule: leakage, keys, identifiers, ...",-394,155 distinct field names
2,"Removed as duplicate information: text labels,...",-417,
3,Fields used to build the crash features (bench...,303,"summarised into 1,456 crash features"


**Interpretation.** The duplicate-information row covers text labels of codes already kept (333), fields repeated from the accident table (61), raw fields replaced by NHTSA's imputed versions (19) and constant fields (4). The remaining rows of the same file (VIN-decoded and post-crash removals) belong to the field selection in notebook 04, Part B.

## 3.4 Constructing the two targets
Both targets come from `MAX_SEV`, the most severe injury in the crash on the KABCO scale (K fatal, A suspected serious, B suspected minor, C possible, O no apparent injury). `MAX_SEV` itself is then removed from the predictors (step 3.3).

* **S1 `y_injury`:** 1 if any injury (C, B, A, K, or injured with unknown severity), 0 if property damage only (O).
* **S2 `y_serious`:** 1 if serious or fatal (A or K), 0 if O, C or B.
* Crashes with unknown or not-applicable severity are set to `-1` and excluded from that target, rather than guessed.

In [6]:
acc24 = pd.read_csv(crss_extracted_dir(2024) / "accident.csv", encoding="latin-1",
                    usecols=["MAX_SEV", "MAX_SEVNAME"], keep_default_na=False)
sev = acc24.MAX_SEV
acc24["y_injury"] = np.select([sev == 0, sev.isin([1, 2, 3, 4, 5])], [0, 1], default=-1)
acc24["y_serious"] = np.select([sev.isin([0, 1, 2]), sev.isin([3, 4])], [0, 1], default=-1)
acc24.groupby(["MAX_SEV", "MAX_SEVNAME", "y_injury", "y_serious"]).size().rename("crashes_2024").reset_index()

,MAX_SEV,MAX_SEVNAME,y_injury,y_serious,crashes_2024
0,0,No Apparent Injury (O),0,0,24732
1,1,Possible Injury (C),1,0,10139
2,2,Suspected Minor Injury (B),1,0,9198
3,3,Suspected Serious Injury (A),1,1,5489
4,4,Fatal Injury (K),1,1,1096
5,5,"Injured, Severity Unknown",1,-1,127
6,6,Died Prior to Crash*,-1,-1,3
7,8,No person involved,-1,-1,17
8,9,Unknown/Not Reported,-1,-1,857


**Interpretation.** The mapping is one-to-one with the official codes. Fewer than 2% of crashes have unknown severity and are excluded from a target rather than assigned a guessed label.

## 3.5 Raw versus imputed fields
For several fields NHTSA publishes both the police-reported value and a statistically **imputed** version (suffix `_IM`) in which missing values have been filled using NHTSA's documented method. Where both exist, the imputed version is kept and the raw one dropped, so missing values are handled consistently with the official national estimates.

For 19 of the 20 imputed fields the raw version is dropped (the 19 "replaced by NHTSA's imputed versions" in the field accounting); for body type (`BDYTYP_IM`) the raw field is kept as well.

In [7]:
reg = pd.read_csv(TABLE_DIR / "crss_feature_register.csv")
imputed_used = sorted({f for f in reg.field if f.endswith("_IM")})
print(len(imputed_used), "imputed fields used:", ", ".join(imputed_used))

20 imputed fields used: AGE_IM, ALCHL_IM, BDYTYP_IM, EJECT_IM, EVENT1_IM, HOUR_IM, IMPACT1_IM, LGTCON_IM, MANCOL_IM, MDLYR_IM, PCRASH1_IM, PERALCH_IM, RELJCT1_IM, RELJCT2_IM, SEAT_IM, SEX_IM, VEVENT_IM, V_ALCH_IM, WEATHR_IM, WKDY_IM


## 3.6 Summarising vehicles and people to one row per crash
Each child-table field is turned into crash-level features:

* **Coded fields** (e.g. body type, rollover, restraint use): the number of vehicles/people in the crash with each frequent code. The list of codes (up to 12 per field, each at least 0.2% of rows) is learned from the **training years 2020-2022 only**, so later years cannot influence the feature design.
* **Numeric fields** (e.g. travel speed, age, model year): the minimum and maximum across the crash, after removing documented "unknown" codes (e.g. speed 998/999).

The example below follows one 2024 crash through this step.

**Why up to 12 codes, and why ignore codes in fewer than 0.2% of training rows?** Without a cap, every code of every field would become its own column, and many would be almost always zero. The check below measures how much of the training data the kept codes cover: across the 233 coded fields in the vehicle- and person-level tables, a median of 99.9% of recorded values; for the 72 fields with more than 12 codes, a median of 98.2% (mean 93.4%). The cap mostly drops rare codes, and a row with a rare code is still counted in its table's row count. (`outputs/tables/crss_code_coverage.csv`, one row per field.)

In [8]:
cov = pd.read_csv(TABLE_DIR / "crss_code_coverage.csv")
print(f"{len(cov)} coded fields; {(cov.distinct_codes > 12).sum()} have more than 12 codes")
print(f"share of training values covered by the kept codes: median {cov.coverage.median():.1%}, "
      f"fields with >12 codes: median {cov[cov.distinct_codes > 12].coverage.median():.1%}")
cov.sort_values("coverage").head(8)

233 coded fields; 72 have more than 12 codes
share of training values covered by the kept codes: median 99.9%, fields with >12 codes: median 98.2%


,table,field,distinct_codes,kept,coverage
37,vehicle,ACC_TYPE,93,12,0.564673
67,parkwork,PHOUR,25,12,0.576010
33,vehicle,P_CRASH2,57,12,0.704328
7,vehicle,MAKE,68,12,0.757263
97,person,MAKE,68,12,0.768095
72,parkwork,PMAKE,57,12,0.768556
142,pbtype,PEDCTYPE,55,12,0.819029
143,pbtype,BIKECTYPE,77,12,0.827238


In [9]:
veh24 = pd.read_csv(crss_extracted_dir(2024) / "vehicle.csv", encoding="latin-1", low_memory=False,
                    usecols=["CASENUM", "VEH_NO", "BODY_TYPNAME", "ROLLOVER", "TRAV_SP", "MOD_YEAR"])
example = veh24[veh24.groupby("CASENUM").VEH_NO.transform("size") == 3].CASENUM.iloc[0]
display(veh24[veh24.CASENUM == example])

features = pd.read_parquet(PROCESSED_DIR / "crss_crash_features.parquet")
row = features[(features.YEAR == 2024) & (features.CASENUM == example)]
cols = ["vehicle__n_rows", "vehicle__TRAV_SP__min", "vehicle__TRAV_SP__max", "vehicle__MDLYR_IM__min",
        "vehicle__ROLLOVER=0"] + [c for c in features.columns if c.startswith("vehicle__BODY_TYP=")][:4]
row[cols].T.rename(columns=lambda _: f"crash {example}")

,CASENUM,VEH_NO,MOD_YEAR,BODY_TYPNAME,TRAV_SP,ROLLOVER
4,202405532158,1,2002,"Compact Utility (Utility Vehicle Categories ""S...",998,0
5,202405532158,2,2015,"Minivan (Chrysler Town and Country, Caravan, G...",998,0
6,202405532158,3,2007,5-door/4-door hatchback,998,0


,crash 202405532158
vehicle__n_rows,3.0
vehicle__TRAV_SP__min,NaN
vehicle__TRAV_SP__max,NaN
vehicle__MDLYR_IM__min,2002.0
vehicle__ROLLOVER=0,3.0
vehicle__BODY_TYP=4,0.0
vehicle__BODY_TYP=14,1.0
vehicle__BODY_TYP=34,0.0
vehicle__BODY_TYP=15,0.0


**Interpretation.** Three vehicle rows become a single crash row: the vehicle count, the speed range, the oldest model year and counts of each body type and rollover code. Travel speeds recorded as unknown (998/999) become missing, not 998 mph.

## 3.7 Verifying the crash table
Checks that the aggregation neither lost nor duplicated crashes, and how the features divide by source.

In [10]:
n_accident = int(inv[inv.table == "accident"].rows.sum())
assert len(features) == n_accident, "row conservation failed"
assert not features.duplicated(["YEAR", "CASENUM"]).any(), "a crash was duplicated"
print(f"crashes: {len(features):,} (accident tables: {n_accident:,}) - conserved, no duplicates")
print(f"features: {len(reg):,}")
display(pd.crosstab(reg.grain, reg.group, margins=True))

crashes: 264,661 (accident tables: 264,661) - conserved, no duplicates
features: 1,456


group,crash_record,post_crash,All
grain,,,
crash,78,0,78
person,368,12,380
vehicle,954,44,998
All,1400,56,1456


**Interpretation.** All 264,661 crashes are kept exactly once, with 1,456 features: 78 crash-level, 998 from vehicle-level tables (383 of them from the VIN-decode specifications) and 380 from person tables.

> **Initial plan:** match fields across the five years by name.
> **What we found:** while compiling the data dictionary, a check of field names across years showed that the 2020 weather table calls its code `WEATHERCode`, while 2021-2024 use `WEATHER` (same codes). Matched by name, 2020 crashes had empty weather-table features and seven duplicate features existed only for 2020.
> **Decision:** rename `WEATHERCode` to `WEATHER` when reading 2020, rebuild the table and retrain. Every other cross-year difference is either capitalisation only (handled by upper-casing all names) or a field NHTSA introduced in a later year (left missing in earlier years). Every feature is also tagged with its source (step 3.8 below), which lets notebook 04 remove whole groups of fields on evidence.

## 3.8 Tagging every feature by source
Each crash feature is tagged by where it comes from. The tags set up the field selection in notebook 04: they make it possible to test or remove a whole group of fields in one step.

In [11]:
import select_crash_features
reg = pd.read_csv(TABLE_DIR / "crss_feature_register.csv")
reg["source"] = np.select([select_crash_features.is_vin(reg), reg.group == "post_crash"],
                          ["VIN-decoded specifications", "Post-crash"], "Police-reported crash information (incl. NHTSA-imputed)")
reg["table_field"] = reg.table + "." + reg.field
reg.groupby("source")[["table_field", "field", "feature"]].apply(
    lambda d: pd.Series({"fields": d.loc[d.field != "n_rows", "table_field"].nunique(), "features": len(d)}))

,fields,features
source,,
Police-reported crash information (incl. NHTSA-imputed),178,993
Post-crash,8,56
VIN-decoded specifications,117,407


**Interpretation.** 178 fields (993 features) are police-reported crash information: vehicles, people, road, light, speed, restraint, alcohol, crash events such as air-bag deployment and ejection, and investigation items such as citations. 117 fields (407 features) are manufacturer specifications decoded from each VIN. 8 fields (56 features) are recorded after the crash: damage, towing and alcohol-test results. (Row counts per table, such as the number of vehicles, are features but not fields.)

## 3.9 Chronological partitions
Splits are by year, not at random, so each model is tested on a later year than it was trained on, which is how it would be used in practice.

In [12]:
split = np.select([features.YEAR.isin(CRSS_TRAIN_YEARS), features.YEAR == CRSS_VALIDATION_YEAR],
                  ["train 2020-22", "validation 2023"], "test 2024")
summary = features.assign(split=split).groupby("split").agg(
    crashes=("CASENUM", "size"),
    injury_rate=("y_injury", lambda s: (s[s >= 0] == 1).mean()),
    serious_rate=("y_serious", lambda s: (s[s >= 0] == 1).mean()))
summary.round(3)

,crashes,injury_rate,serious_rate
split,,,
test 2024,51658,0.513,0.130
train 2020-22,162900,0.531,0.136
validation 2023,50103,0.515,0.135


**Interpretation.** Target rates are stable across the three periods (about 51-53% injury, about 13% serious or fatal), so differences in test results reflect the models rather than a shift in the outcome. Validation (2023) is used for all tuning and threshold choices; 2024 is scored once at the end.

# Part B - Owner complaints (unstructured narratives + metadata)

## 3.10 Reading the flat file correctly
The complaint file is tab-delimited and **unquoted**: narratives may contain literal `"` characters.

> **Initial plan:** read the file with standard CSV parsing (as in the first raw-data audit).
> **What we found:** standard parsing treats a `"` inside a narrative as the start of a quoted field and merges neighbouring records. The first audit therefore reported 2,249,110 rows and two "malformed" records.
> **Decision:** read with quoting disabled (`csv.QUOTE_NONE`). Every one of the **2,250,304** rows then has the documented 51 fields, matching the official count; the file's SHA-256 hash confirms the source was never altered.

The cell below reproduces the problem on 20 physical lines around line 556,048, where the default parser first breaks (a full-file scan located it).

In [13]:
with open(complaint_file(), encoding="latin-1", newline="") as fh:
    lines = [line for i, line in zip(range(556_060), fh) if i >= 556_040]
window = "".join(lines)
default = list(csv.reader(io.StringIO(window), delimiter="	"))
correct = list(csv.reader(io.StringIO(window), delimiter="	", quoting=csv.QUOTE_NONE))
print(f"lines read      : {len(lines)}")
print(f"default quoting : {len(default)} records, field counts {sorted({len(r) for r in default})}")
print(f"QUOTE_NONE      : {len(correct)} records, field counts {sorted({len(r) for r in correct})}")

lines read      : 20
default quoting : 16 records, field counts [51, 72]
QUOTE_NONE      : 20 records, field counts [51]


**Interpretation.** With default quoting, an unbalanced `"` in one narrative makes the parser read on past the end of the line and swallow the following lines into one record (here 5 physical lines become one 72-field record), so records disappear and field counts break. With quoting disabled, every line is exactly one 51-field record. Applied to the whole file, this is the difference between 2,249,110 and 2,250,304 rows.

The rest of Part B uses the first 300,000 records, read correctly, for illustrations.

In [14]:
with open(complaint_file(), encoding="latin-1", newline="") as fh:
    head = "".join(next(fh) for _ in range(300_000))
correct = list(csv.reader(io.StringIO(head), delimiter="	", quoting=csv.QUOTE_NONE))
print(f"{len(correct):,} records read for illustration")

300,000 records read for illustration


## 3.11 Scope: vehicle complaints only
The file also contains tyre (`T`), child-seat (`C`) and equipment (`E`) products, which use different component lists. The project keeps vehicle complaints (`V`), 96.8% of rows.

In [15]:
audit = json.loads((TABLE_DIR / "feasibility_summary.json").read_text())
pd.Series(audit["complaints"]["prod_type_counts"], name="rows").rename(
    {"V": "V vehicle", "T": "T tyre", "C": "C child seat", "E": "E equipment", "": "(blank)"}).to_frame()

,rows
V vehicle,2178026
T tyre,41571
C child seat,15358
E equipment,15330
(blank),19


## 3.12 One row per complaint
A complaint naming several components appears once per component, all sharing the same `ODINO`. Rows are therefore grouped by `ODINO`: the narrative and vehicle are kept once, the components become a list and the crash/fire/injury/death flags are combined.

In [16]:
raw = pd.DataFrame(correct, columns=complaint_build.COMPLAINT_COLUMNS)
multi = raw[raw.groupby("ODINO").ODINO.transform("size") >= 3].ODINO.iloc[0]
raw.loc[raw.ODINO == multi, ["ODINO", "MAKETXT", "MODELTXT", "YEARTXT", "COMPDESC", "CRASH", "INJURED"]]

,ODINO,MAKETXT,MODELTXT,YEARTXT,COMPDESC,CRASH,INJURED
7,958244,PLYMOUTH,VOYAGER,1989,STRUCTURE:BODY:DOOR:HINGE AND ATTACHMENTS,N,0
8,958244,PLYMOUTH,VOYAGER,1989,STRUCTURE:BODY:DOOR,N,0
9,958244,PLYMOUTH,VOYAGER,1989,VISIBILITY:WINDSHIELD WIPER/WASHER,N,0


## 3.13 Harmonising component names
`COMPDESC` names changed across NHTSA taxonomy revisions (e.g. "ENGINE AND ENGINE COOLING" became "ENGINE"; "SERVICE BRAKES, HYDRAULIC" became "SERVICE BRAKES"; driver-assistance categories were added). 55 top-level names are mapped to **19 component groups**; "UNKNOWN OR OTHER" and non-component entries are not used as labels.

**How the mapping was made.** Each `COMPDESC` is a path such as "SERVICE BRAKES, HYDRAULIC: FOUNDATION COMPONENTS". The first level (before the colon) gives the 55 top-level names. Names that describe the same system were grouped by hand into a fixed list (`GROUPS` in `src/build_complaints.py`); the table below shows every name, its group and its number of rows, and the same table is in the data dictionary (Component mapping sheet).

In [17]:
mapping = pd.read_csv(TABLE_DIR / "complaint_component_mapping.csv", keep_default_na=False).replace({"component_l1": {"": "(blank)"}})
mapping.groupby("group").agg(source_names=("component_l1", lambda s: "; ".join(s)), rows=("rows", "sum")) \
       .sort_values("rows", ascending=False)

,source_names,rows
group,,
ENGINE,ENGINE; ENGINE AND ENGINE COOLING,291137
ELECTRICAL,ELECTRICAL SYSTEM; COMMUNICATION; COMMUNICATIONS,249305
POWER TRAIN,POWER TRAIN,218952
BRAKES,"SERVICE BRAKES, HYDRAULIC; SERVICE BRAKES; PAR...",198010
AIR BAGS,AIR BAGS,156485
STEERING,STEERING,144306
FUEL/PROPULSION,"FUEL/PROPULSION SYSTEM; FUEL SYSTEM, GASOLINE;...",128508
(not a component: excluded from U1 labels),UNKNOWN OR OTHER; CHILD SEAT; FIRERELATED; CHE...,121353
STRUCTURE,STRUCTURE,104217


> **Initial plan:** treat component routing as single-label (one component per complaint).
> **What we found:** 22.6% of complaints name more than one component group.
> **Decision:** U1 is **multi-label**: each complaint keeps all its groups, and the model may recommend more than one.

## 3.14 Targets and narrative cleaning
* **U1:** the complaint's component groups (19 yes/no columns, `c__<GROUP>`).
* **U2 `y_serious`:** 1 if the complaint's structured fields report a crash, fire, injury or death. These flags are entered on the complaint form, separately from the narrative, so they are a genuine label for the text.

NHTSA editorial artefacts are removed from narratives: editor initials such as `*TR` and standard FOIA redaction notices. They carry no information about the vehicle, and editor initials would let a model learn *who processed* a complaint (and when) instead of what it says.

After cleaning, narratives **under 20 characters** are dropped, because they carry too little text to classify.

In [18]:
sample = raw.CDESCR[raw.CDESCR.str.contains(r"\*[A-Z]{2,3}\b", regex=True)].iloc[0]
print("BEFORE:", sample[-160:])
print("AFTER: ", complaint_build.ARTEFACTS.sub(" ", sample)[-160:].strip())

BEFORE: FUEL LEAKED FROM FUEL TANK AREA, EMITTING STRONG FUMES IN GARAGE.  *AK
AFTER:  FUEL LEAKED FROM FUEL TANK AREA, EMITTING STRONG FUMES IN GARAGE.


## 3.15 Duplicates and chronological split
Exact duplicate narratives (after normalising case and punctuation) are removed, keeping the earliest copy, so no validation or test narrative is a verbatim copy of a training one. Complaints are then split by the date NHTSA received them.

In [19]:
cm = pd.read_parquet(PROCESSED_DIR / "complaints_model.parquet",
                     columns=["split", "year", "y_serious", "components"])
split_summary = pd.read_csv(TABLE_DIR / "complaint_split_summary.csv").set_index("split").loc[["train", "validation", "test"]]
split_summary[["complaints", "first", "last", "serious_rate", "multi_component", "exact_duplicates_removed"]]

,complaints,first,last,serious_rate,multi_component,exact_duplicates_removed
split,,,,,,
train,1260948,1995-01-01,2021-12-31,0.103392,0.191889,17721
validation,114501,2022-01-01,2023-12-31,0.063938,0.243570,927
test,194825,2024-01-01,2026-09-24,0.055645,0.261116,3777


> **Initial plan:** train the text models on every complaint received up to 2021 (1995-2021).
> **What we found:** complaints before 2010 use the older component taxonomy (step 3.12) and an older narrative style, and the component mix has shifted strongly over time (driver-assistance complaints rose from about 2% to 10%).
> **Decision:** the text models train on complaints received **2010-2021**. The earlier complaints stay in the prepared table and in the exploratory analysis.

In [20]:
train_window = cm[(cm.split == "train") & (cm.year >= 2010)]
print(f"training complaints 1995-2021: {(cm.split == 'train').sum():,}")
print(f"training complaints 2010-2021: {len(train_window):,}  <- used for U1/U2")
print(f"validation 2022-2023:          {(cm.split == 'validation').sum():,}")
print(f"test 2024-2026:                {(cm.split == 'test').sum():,}")

training complaints 1995-2021: 1,260,948
training complaints 2010-2021: 741,623  <- used for U1/U2
validation 2022-2023:          114,501
test 2024-2026:                194,825


## 3.16 Summary and hand-off to Modelling
| Table | Rows | Predictors | Targets | Split |
|---|---|---|---|---|
| CRSS crash table | 264,661 crashes | 1,456 features from 303 fields in all 28 tables, each tagged by source (police-reported 993, VIN-decoded 407, post-crash 56) | `y_injury`, `y_serious` | train 2020-22 / validation 2023 / test 2024 |
| Complaint table | 1,570,274 complaints | narrative text (TF-IDF is fitted in notebook 04, on training text only) | 19 component groups, `y_serious` | train 2010-21 / validation 2022-23 / test 2024-26 |

Every encoding and vocabulary that learns from data is fitted on the training period only. Notebook 04 trains and compares the benchmark models on all of these features, then loops back to this phase to select the fields the final crash models use.